# Note for Vy~

panda = grafik data

numpy = math

train_test_split = pisah buat train sama test

RandomForestClassifier = metode yang dipake ( Random Fores Classifier )

Pickle = Buat save as pkl

drive = buat ambil data dari google drive

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle as pkl
from google.colab import drive

SIapin datanya dulu -->

In [ ]:
drive.mount('/content/drive')

filepath = '/content/drive/MyDrive/Verdy akbar r/AOL AI/data_balita.csv'

df = pd.read_csv(filepath)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Check Data Information


In [ ]:
print("First 5 rows of the DataFrame:")
print(df.head())

print("\nDataFrame Info (data types and non-null counts):")
df.info()

print("\nDescriptive statistics of numerical columns:")
print(df.describe())

First 5 rows of the DataFrame:
   Umur (bulan) Jenis Kelamin  Tinggi Badan (cm)       Status Gizi
0             0     laki-laki          44.591973           stunted
1             0     laki-laki          56.705203            tinggi
2             0     laki-laki          46.863358            normal
3             0     laki-laki          47.508026            normal
4             0     laki-laki          42.743494  severely stunted

DataFrame Info (data types and non-null counts):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120999 entries, 0 to 120998
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Umur (bulan)       120999 non-null  int64  
 1   Jenis Kelamin      120999 non-null  object 
 2   Tinggi Badan (cm)  120999 non-null  float64
 3   Status Gizi        120999 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 3.7+ MB

Descriptive statistics of numerical columns:
     

## Check for Missing Values

In [ ]:
print("Number of missing values per column:")
print(df.isnull().sum())

Number of missing values per column:
Umur (bulan)         0
Jenis Kelamin        0
Tinggi Badan (cm)    0
Status Gizi          0
dtype: int64


## Identify and Handle Outliers


In [ ]:
numerical_cols = ['Umur (bulan)', 'Tinggi Badan (cm)']

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
    print(f"Column '{col}':")
    print(f"  Q1: {Q1:.2f}")
    print(f"  Q3: {Q3:.2f}")
    print(f"  IQR: {IQR:.2f}")
    print(f"  Lower Bound: {lower_bound:.2f}")
    print(f"  Upper Bound: {upper_bound:.2f}")
    print(f"  Number of outliers: {outliers_count}")

    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    print(f"  Outliers capped in '{col}'.\n")

print("DataFrame after outlier capping (first 5 rows):")
print(df.head())

Column 'Umur (bulan)':
  Q1: 15.00
  Q3: 45.00
  IQR: 30.00
  Lower Bound: -30.00
  Upper Bound: 90.00
  Number of outliers: 0
  Outliers capped in 'Umur (bulan)'.

Column 'Tinggi Badan (cm)':
  Q1: 77.00
  Q3: 101.20
  IQR: 24.20
  Lower Bound: 40.70
  Upper Bound: 137.50
  Number of outliers: 38
  Outliers capped in 'Tinggi Badan (cm)'.

DataFrame after outlier capping (first 5 rows):
   Umur (bulan) Jenis Kelamin  Tinggi Badan (cm)       Status Gizi
0           0.0     laki-laki          44.591973           stunted
1           0.0     laki-laki          56.705203            tinggi
2           0.0     laki-laki          46.863358            normal
3           0.0     laki-laki          47.508026            normal
4           0.0     laki-laki          42.743494  severely stunted


## Manual Model Testing

In [ ]:
umur = float(input("Enter Umur (bulan): "))
tinggi_badan = float(input("Enter Tinggi Badan (cm): "))
jenis_kelamin_str = input("Enter Jenis Kelamin (laki-laki/perempuan): ")

gender_mapping = {'laki-laki': 0, 'perempuan': 1}
jenis_kelamin_encoded = gender_mapping.get(jenis_kelamin_str.lower())

if jenis_kelamin_encoded is None:
    print("Invalid Jenis Kelamin. Please enter 'laki-laki' or 'perempuan'.")
else:
    input_data_for_prediction = pd.DataFrame([{
        'Umur (bulan)': umur,
        'Tinggi Badan (cm)': tinggi_badan,
        'Jenis Kelamin': jenis_kelamin_encoded
    }])

    prediction_numeric = model.predict(input_data_for_prediction)

    prediction_label = le.inverse_transform(prediction_numeric)

    print(f"\nPredicted Status Gizi: {prediction_label[0]}")

Enter Umur (bulan): 12
Enter Tinggi Badan (cm): 160
Enter Jenis Kelamin (laki-laki/perempuan): laki-laki

Predicted Status Gizi: tinggi


/tmp/ipython-input-3524238719.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  input_data_processed = pd.concat([all_features, input_data_processed], ignore_index=True)
/tmp/ipython-input-3524238719.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input_data_processed = input_data_processed.fillna(False) # Fill new one-hot columns with False


## Check and Clean Numerical Columns


In [ ]:
print("Data types and non-null counts for 'Umur (bulan)' and 'Tinggi Badan (cm)':")
df[['Umur (bulan)', 'Tinggi Badan (cm)']].info()

print("\nFirst 5 rows of 'Umur (bulan)' and 'Tinggi Badan (cm)' for visual inspection:")
print(df[['Umur (bulan)', 'Tinggi Badan (cm)']].head())

print("\nDescriptive statistics for 'Umur (bulan)' and 'Tinggi Badan (cm)':")
print(df[['Umur (bulan)', 'Tinggi Badan (cm)']].describe())

Data types and non-null counts for 'Umur (bulan)' and 'Tinggi Badan (cm)':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120999 entries, 0 to 120998
Data columns (total 2 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Umur (bulan)       120999 non-null  float64
 1   Tinggi Badan (cm)  120999 non-null  float64
dtypes: float64(2)
memory usage: 1.8 MB

First 5 rows of 'Umur (bulan)' and 'Tinggi Badan (cm)' for visual inspection:
   Umur (bulan)  Tinggi Badan (cm)
0           0.0          44.591973
1           0.0          56.705203
2           0.0          46.863358
3           0.0          47.508026
4           0.0          42.743494

Descriptive statistics for 'Umur (bulan)' and 'Tinggi Badan (cm)':
        Umur (bulan)  Tinggi Badan (cm)
count  120999.000000      120999.000000
mean       30.173803          88.655552
std        17.575119          17.300667
min         0.000000          40.700000
25%        15.000000  

## Explicitly Map Gender


In [ ]:
gender_mapping = {'laki-laki': 0, 'perempuan': 1}

df['Jenis Kelamin'] = df['Jenis Kelamin'].map(gender_mapping)

print("First 5 rows of 'Jenis Kelamin' after mapping:")
print(df['Jenis Kelamin'].head())

print("\nData type of 'Jenis Kelamin' after mapping:")
print(df['Jenis Kelamin'].dtype)

First 5 rows of 'Jenis Kelamin' after mapping:
0    0
1    0
2    0
3    0
4    0
Name: Jenis Kelamin, dtype: int64

Data type of 'Jenis Kelamin' after mapping:
int64


## Explicitly Map Target Variable



In [ ]:
status_gizi_mapping = {
    'normal': 0,
    'tinggi': 1,
    'stunted': 2,
    'severely stunted': 3
}

df['Status Gizi'] = df['Status Gizi'].map(status_gizi_mapping)
y = df['Status Gizi']

# LabelEncoder was already imported in the initial setup cell.
le = LabelEncoder()

# Ensure le.classes_ is set correctly even if not fit on original string labels
# This makes inverse_transform work as expected with the manual mapping.
le.classes_ = np.array(list(status_gizi_mapping.keys()))

print("First 5 entries of transformed 'Status Gizi' (y):")
print(y.head())
print("\nData type of 'Status Gizi' (y) after mapping:")
print(y.dtype)
print("\nLabelEncoder classes:", le.classes_)

First 5 entries of transformed 'Status Gizi' (y):
0    2
1    1
2    0
3    0
4    3
Name: Status Gizi, dtype: int64

Data type of 'Status Gizi' (y) after mapping:
int64

LabelEncoder classes: ['normal' 'tinggi' 'stunted' 'severely stunted']


In [ ]:
X = df[['Umur (bulan)', 'Tinggi Badan (cm)', 'Jenis Kelamin']]
y = df['Status Gizi']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)
print("\nFirst 5 rows of X_train after preprocessing:")
print(X_train.head())

Shape of X_train: (96799, 3)
Shape of X_test: (24200, 3)
Shape of y_train: (96799,)
Shape of y_test: (24200,)

First 5 rows of X_train after preprocessing:
       Umur (bulan)  Tinggi Badan (cm)  Jenis Kelamin
21452          11.0               64.0              0
25283          13.0               79.6              0
30909          15.0               67.8              1
64536          32.0               85.3              1
68658          34.0               79.0              1


## Train RandomForestClassifier Model

In [ ]:
model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

print("RandomForestClassifier model trained successfully with new explicit mappings.")

RandomForestClassifier model trained successfully with new explicit mappings.


## Evaluate Model Performance


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

# accuracy score
print(f"Model Accuracy on the test set with explicit mappings: {accuracy:.4f}")

importances = model.feature_importances_
print(f"Importance - Age: {importances[0]:.2f}, Height: {importances[1]:.2f}, Gender: {importances[2]:.2f}")

if importances[1] < 0.1:
    print("WARNING: Model is STILL ignoring height. Check your CSV data.")
else:
    print("SUCCESS: Model is now using Height correctly!")

Model Accuracy on the test set with explicit mappings: 0.9991
Importance - Age: 0.42, Height: 0.58, Gender: 0.00
SUCCESS: Model is now using Height correctly!


## Save the Trained Model and Target Variable Mapping

In [ ]:
# 1. Define a filename for the saved model
model_filename = 'GrowthForecast_explicit_mapping.pkl'

# 2. Use pickle.dump() to save the trained model to the specified file
with open(model_filename, 'wb') as file:
    pkl.dump(model, file)

print(f"Trained model successfully saved to '{model_filename}'")

# 3. Define a filename for the saved LabelEncoder
le_filename = 'label_encoder_explicit_mapping.pkl'

# 4. Use pickle.dump() to save the le instance to its specified file
with open(le_filename, 'wb') as file:
    pkl.dump(le, file)

print(f"LabelEncoder successfully saved to '{le_filename}'")

Trained model successfully saved to 'GrowthForecast_explicit_mapping.pkl'
LabelEncoder successfully saved to 'label_encoder_explicit_mapping.pkl'
